## 02: Preprocessing
Merge phenotype + environment + genomic data into one flat feature matrix ready for modeling.

Flip `SAMPLE_MODE = False` once ready to run on full data.

In [1]:
import pandas as pd
import numpy as np
import os
import re

SAMPLE_MODE = True  #flip to False for full run

#paths
if SAMPLE_MODE:
    PHENO_PATH = '../data/raw/sample_data/sample_C1_phenotype_100rows.csv'
    ENV_PATH   = '../data/raw/sample_data/sample_environmental_20rows.csv'
    def geno_path(cluster_id, pop_num):
        return f'../data/raw/sample_data/sample_C{cluster_id}.{pop_num}_100rows.csv'
else:
    PHENO_PATH = '../data/raw/C1_Phenotype_Data_V2.csv'
    ENV_PATH   = '../data/raw/environmental_features.csv'
    def geno_path(cluster_id, pop_num):
        return f'../data/raw/genotypes/C{cluster_id}/C{cluster_id}.{pop_num}_Imputed.csv'

OUTPUT_PATH = '../data/processed/merged_sample.csv' if SAMPLE_MODE else '../data/processed/merged_full.csv'
print('SAMPLE_MODE:', SAMPLE_MODE)


SAMPLE_MODE: True


### 1. Load raw data

In [2]:
pheno = pd.read_csv(PHENO_PATH)
env   = pd.read_csv(ENV_PATH)
print('Raw pheno:', pheno.shape)
print('Env:      ', env.shape)


Raw pheno: (100, 33)
Env:       (20, 86)


### 2. Clean phenotype
Drop junk columns left over from the pre-merge, normalize the YEAR column name, and parse `LINE_UNIQUE_ID` into its three components.

In [3]:
JUNK_COLS = [
    'Unnamed: 0', 'Unnamed: 0_x', 'Unnamed: 0_y',
    'projects_x', 'projects_y',
    'FILE_LIST', 'shorthand_x', 'shorthand_y',
    'YEAR_y', 'projectID', 'MAB_PROJECT_ID', 'GENERATION_NAME',
]
pheno = pheno.drop(columns=[c for c in JUNK_COLS if c in pheno.columns])

# normalize YEAR (sample pheno has YEAR_x due to pre-merge duplication)
if 'YEAR_x' in pheno.columns:
    pheno = pheno.rename(columns={'YEAR_x': 'YEAR'})

# parse LINE_UNIQUE_ID: 'C1.1.191' -> cluster=1, pop=1, line=191
parsed = pheno['LINE_UNIQUE_ID'].str.extract(r'^C(\d+)\.(\d+)\.(\d+)$')
parsed.columns = ['CLUSTER_ID', 'POP_NUM', 'LINE_NUM']
pheno[['CLUSTER_ID', 'POP_NUM', 'LINE_NUM']] = parsed
pheno['POP_NUM']  = pheno['POP_NUM'].astype(int)
pheno['LINE_NUM'] = pheno['LINE_NUM'].astype(int)

n_unparsed = pheno['LINE_NUM'].isnull().sum()
print(f'Clean pheno: {pheno.shape}  |  LINE_UNIQUE_ID parse failures: {n_unparsed}')
pheno[['LINE_UNIQUE_ID', 'CLUSTER_ID', 'POP_NUM', 'LINE_NUM']].head(3)


Clean pheno: (100, 24)  |  LINE_UNIQUE_ID parse failures: 0


,LINE_UNIQUE_ID,CLUSTER_ID,POP_NUM,LINE_NUM
0,C1.1.191,1,1,191
1,C1.1.193,1,1,193
2,C1.1.62,1,1,62


### 3. Merge phenotype + environment
Left join on `YEAR + LOC` so every phenotype row keeps its env features (or gets NaN if the env row is missing — expected in sample mode).

In [4]:
pheno_env = pheno.merge(env, on=['YEAR', 'LOC'], how='left')
print(f'Pheno+Env shape: {pheno_env.shape}')

env_feat_cols = [c for c in env.columns if c not in ('YEAR', 'LOC')]
rows_no_env = pheno_env[env_feat_cols].isnull().all(axis=1).sum()
print(f'Rows with no env match: {rows_no_env}/{len(pheno_env)} (expected ~all in sample mode)')


Pheno+Env shape: (100, 108)
Rows with no env match: 100/100 (expected ~all in sample mode)


### 4. Merge with genomic data
For each (cluster, population) group in the phenotype, load the matching genomic file, filter to progeny rows only (dropping parent PIDs), and left-join on `LINE_NUM`.

In [5]:
def load_geno(cluster_id, pop_num):
    """Load one imputed genomic file; return progeny rows with LINE_NUM column."""
    path = geno_path(cluster_id, pop_num)
    if not os.path.exists(path):
        print(f'  [MISS] {path}')
        return None
    geno = pd.read_csv(path, index_col=0)
    # keep only 11-digit zero-padded progeny rows (drop PID... parent rows)
    progeny_mask = geno.index.str.match(r'^\d{11}$')
    geno = geno.loc[progeny_mask].copy()
    geno['LINE_NUM'] = geno.index.astype(int)
    return geno

all_merged = []
groups = pheno_env.groupby(['CLUSTER_ID', 'POP_NUM'], sort=False)
print(f'Populations to process: {groups.ngroups}')

for (cluster_id, pop_num), group in groups:
    geno = load_geno(cluster_id, int(pop_num))
    if geno is not None:
        merged = group.merge(geno.drop(columns=['LINE_NUM']).assign(LINE_NUM=geno['LINE_NUM']),
                             on='LINE_NUM', how='left')
        print(f'  C{cluster_id}.{pop_num}: {len(group)} pheno rows -> {merged.shape[1]} cols after geno merge')
    else:
        merged = group.copy()
    all_merged.append(merged)

df = pd.concat(all_merged, ignore_index=True)
print(f'\nFinal merged shape: {df.shape}')


Populations to process: 1
  C1.1: 100 pheno rows -> 3019 cols after geno merge

Final merged shape: (100, 3019)


### 5. Quality check

In [6]:
# Target variable
print(f'YLD_BE missing: {df["YLD_BE"].isnull().sum()}/{len(df)}')

# SNP missingness (pre-imputed files should be ~0%)
snp_cols = [c for c in df.columns if c.startswith('M')]
if snp_cols:
    snp_null_pct = df[snp_cols].isnull().mean().mean() * 100
    print(f'SNP cols: {len(snp_cols)}  |  mean missingness: {snp_null_pct:.1f}%')
else:
    print('No SNP columns merged (check geno file match)')

# Env missingness
env_null_pct = df[env_feat_cols].isnull().mean().mean() * 100
print(f'Env feat missingness: {env_null_pct:.1f}%')

# Preview
key_cols = ['LINE_UNIQUE_ID', 'YEAR', 'LOC', 'YLD_BE'] + snp_cols[:3] + env_feat_cols[:3]
df[[c for c in key_cols if c in df.columns]].head(3)


YLD_BE missing: 3/100
SNP cols: 2912  |  mean missingness: 65.5%
Env feat missingness: 100.0%


,LINE_UNIQUE_ID,YEAR,LOC,YLD_BE,MST,M00003409443,M00000005000,X04_PRCP,X05_PRCP,X06_PRCP
0,C1.1.191,2001,NEDA,156.052,19.7,NaN,NaN,NaN,NaN,NaN
1,C1.1.193,2001,NEDA,150.685,20.2,NaN,NaN,NaN,NaN,NaN
2,C1.1.62,2001,IAPR,157.218,19.1,1.0,1.0,NaN,NaN,NaN


### 6. Save

In [7]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved -> {OUTPUT_PATH}  ({os.path.getsize(OUTPUT_PATH)/1024:.0f} KB)')


Saved -> ../data/processed/merged_sample.csv  (682 KB)


SAMPLE DATA INTERP:

The two 100% missingness numbers are both sample artifacts. The env miss at 100% is the same LOC mismatch seen in the 01_eda file. The SNP miss at 65.5% comes directly from our EDA finding that the sample geno file covers lines 1-100, but pheno lines mostly run 62-198, so only line 62 and a handful of low-numbered lines matched. Every unmatched row gets NaN across all 2912 SNP columns. That math lands us right around 60-65% missingness. Row 2 on line 62 shows actual SNP values (1.0, 1.0) while rows 0-1 (lines 191, 193) are all NaN.

When we flip to SAMPLE_MODE = False:

Env missingness should drop to near 0% (full env file covers all LOC+YEAR combos)
SNP missingness should drop to near 0% (full geno files are pre-imputed, and every pheno line will have a matching geno file)
The geno loop will spin over hundreds of populations instead of 1, so expect a few minutes of runtime

The pipeline is structurally correct & The 3/100 missing YLD_BE is the only real data gap, and those 3 rows need to be dropped before modeling since that's the target.